# PoseCoach P01 — Dataset Prep and YOLO26 Finetune (Kaggle Edition)
> **Run on:** Kaggle Notebook — GPU T4 x2 (30 free hrs/week) | **Storage:** /kaggle/working/
>
> Based on [Ultralytics YOLO26 Fine-Tuning Guide](https://docs.ultralytics.com/guides/finetuning-guide/)
> and [YOLO26 Training Recipe](https://docs.ultralytics.com/guides/yolo26-training-recipe/)
>
> Uses **two-stage training**: freeze backbone → unfreeze all with lower LR
> Uses **dual-dataset strategy**: Kaggle Workout + Fit3D
> Evaluates **all four thesis metrics**: OKS-mAP, MAE ≤5°, CPU latency, rep accuracy

### How this differs from the Colab version
- No Google Drive mount — outputs live in `/kaggle/working/` automatically
- No `kaggle.json` upload — credentials are pre-configured inside Kaggle
- Dataset attached via the **Data tab** (right panel) — no CLI download needed
- No keep-alive cell needed — Kaggle sessions don't idle-disconnect
- Sessions run up to 9 hrs; **resume logic in Step 9 handles restarts automatically**

### Steps
1. Set paths & verify GPU
2. Confirm Kaggle credentials
3. Install dependencies
4. Locate & prepare dataset
5. Extract frames at 2 FPS
6. Extract YOLO26 keypoints
7. Prepare YOLO pose dataset
8. Baseline evaluation
9. Two-stage fine-tune (auto-resume on restart)
10. Save weights & evaluate | 10b. ONNX + latency | 10c. Angle MAE
11. Fit3D golden angle templates
12. Rep counter validation

## ✅ Step 1 — Verify GPU & Set Paths

In [ ]:
import torch, os

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f"GPU: {gpu_name}")
if gpu_name == 'NO GPU':
    print("  ⚠️  Switch to GPU: Settings → Accelerator → GPU T4 x2")
print(f"CUDA: {torch.version.cuda}")

# ── Kaggle paths ──
KAGGLE_ROOT = '/kaggle/working/gymvision'
DATASET_DIR = f'{KAGGLE_ROOT}/datasets/workoutfitness'
FRAMES_DIR  = f'{KAGGLE_ROOT}/datasets/frames'
KP_DIR      = f'{KAGGLE_ROOT}/datasets/keypoints'
YOLO_DIR    = f'{KAGGLE_ROOT}/datasets/yolo_pose'
MODELS_DIR  = f'{KAGGLE_ROOT}/models'
DRIVE_ROOT  = KAGGLE_ROOT  # alias — rest of notebook uses DRIVE_ROOT

for d in [DATASET_DIR, FRAMES_DIR, KP_DIR, YOLO_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'\n✅ Paths ready under {KAGGLE_ROOT}')

## ✅ Step 2 — Kaggle Credentials (auto-configured, nothing to do)

In [ ]:
import subprocess
r = subprocess.run(['kaggle', 'datasets', 'list', '--max-size', '1'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ Kaggle credentials working')
else:
    print('⚠️  kaggle CLI issue:', r.stderr[:200])
    print('   Make sure Internet is ON: Settings → Internet → On')

## ✅ Step 3 — Install Dependencies

In [ ]:
%%capture
!pip install ultralytics kaggle opencv-python-headless numpy tqdm scipy

## ✅ Step 4 — Locate & Prepare Dataset

In [ ]:
import os

# ── Dataset is mounted here by Kaggle ──
DATASET_SOURCE = '/kaggle/input/datasets/hasyimabdillah/workoutfitness-video'

if not os.path.exists(DATASET_SOURCE):
    print('❌ Dataset not found. Add it via: Data tab → + Add Input → workoutfitness-video')
    raise RuntimeError('Dataset missing.')

print(f'✅ Dataset found: {DATASET_SOURCE}')
exercises = sorted(os.listdir(DATASET_SOURCE))
print(f'   {len(exercises)} exercise folders: {exercises}')

# ── Symlink into working dir so all downstream cells use DATASET_DIR ──
os.makedirs(DATASET_DIR, exist_ok=True)
linked = 0
for item in exercises:
    src_p = os.path.join(DATASET_SOURCE, item)
    dst_p = os.path.join(DATASET_DIR, item)
    if not os.path.exists(dst_p):
        os.symlink(src_p, dst_p)
        linked += 1

print(f'\n✅ Linked {linked} folder(s) into {DATASET_DIR}')
print('   Ready for Step 5 — frame extraction')

## ✅ Step 5 — Extract Frames at 2 FPS

In [ ]:
import cv2, os
from tqdm import tqdm

def extract_frames(video_path, out_dir, fps=2):
    cap = cv2.VideoCapture(video_path)
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    interval = max(1, int(video_fps / fps))
    os.makedirs(out_dir, exist_ok=True)
    count = saved = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if count % interval == 0:
            cv2.imwrite(f'{out_dir}/frame_{saved:06d}.jpg', frame,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved += 1
        count += 1
    cap.release()
    return saved

# ── os.walk with followlinks=True — required because DATASET_DIR uses symlinks ──
video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
video_files = []
for root, dirs, files in os.walk(DATASET_DIR, followlinks=True):
    for f in files:
        if os.path.splitext(f)[1].lower() in video_exts:
            video_files.append(os.path.join(root, f))

print(f'Found {len(video_files)} video files')

total_frames = 0
for vid in tqdm(video_files, desc='Extracting frames'):
    parts = vid.replace(DATASET_DIR + '/', '').split('/')
    exercise = parts[0]
    clip_id = os.path.splitext(parts[-1])[0]
    out = f'{FRAMES_DIR}/{exercise}/{clip_id}'
    if os.path.exists(out) and len(os.listdir(out)) > 0:
        total_frames += len(os.listdir(out))
        continue
    total_frames += extract_frames(vid, out, fps=2)

print(f'\n✅ Extracted {total_frames:,} frames total')
for ex in sorted(os.listdir(FRAMES_DIR)):
    clips = os.listdir(f'{FRAMES_DIR}/{ex}')
    frames = sum(len(os.listdir(f'{FRAMES_DIR}/{ex}/{c}')) for c in clips)
    print(f'  {ex}: {len(clips)} clips, {frames} frames')

## ✅ Step 6 — Extract YOLO26 Keypoints

In [ ]:
import numpy as np
from ultralytics import YOLO

# YOLO26n-pose — NMS-free end-to-end (DO NOT add NMS after predict)
model = YOLO('yolo26n-pose.pt')
model.to('cuda' if torch.cuda.is_available() else 'cpu')

frame_paths = list(Path(FRAMES_DIR).rglob('*.jpg'))
print(f'Processing {len(frame_paths):,} frames for keypoints...')

skipped = saved = 0
for img_path in tqdm(frame_paths, desc='Extracting keypoints'):
    rel = img_path.relative_to(FRAMES_DIR)
    kp_path = Path(KP_DIR) / rel.with_suffix('.npy')
    kp_path.parent.mkdir(parents=True, exist_ok=True)
    if kp_path.exists():
        skipped += 1
        continue
    results = model.predict(str(img_path), verbose=False, conf=0.3)
    if results[0].keypoints is None or len(results[0].keypoints) == 0:
        continue
    # CRITICAL: use .xyn (normalized) NOT .xy
    kp = results[0].keypoints.xyn.cpu().numpy()
    np.save(str(kp_path), kp)
    saved += 1

print(f'\n✅ Keypoints saved: {saved:,} | Skipped (cached): {skipped:,}')

## ✅ Step 7 — Prepare YOLO Pose Dataset (80/20 Stratified Split)

In [ ]:
import shutil, random, yaml
from collections import defaultdict

random.seed(42)

# IMPORTANT: nc=1 (person class ONLY) for pose estimation.
# Why NOT nc=7?
# 1. yolo26n-pose.pt pretrained with nc=1. Changing nc reinitializes detection head.
# 2. Single-frame exercise classification is unreliable.
# 3. Classification competes with keypoints for model capacity.
# Exercise classification should be done SEPARATELY (user selects, or DTW/LSTM on sequence).

FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

EXERCISE_KEYWORDS = ['squat', 'deadlift', 'curl', 'bench', 'ohp', 'lunge', 'plank',
                     'press', 'row', 'pullup', 'pushup', 'crunch', 'extension']

all_clips = []
for img_path in Path(FRAMES_DIR).rglob('*.jpg'):
    exercise = img_path.parent.parent.name.lower()
    clip_id = img_path.parent.name
    matched = any(k in exercise for k in EXERCISE_KEYWORDS)
    if matched:
        all_clips.append(clip_id)

all_clips = list(set(all_clips))
random.shuffle(all_clips)
split = max(1, int(len(all_clips) * 0.8))
train_clips = set(all_clips[:split])
val_clips = set(all_clips[split:])
print(f'Total clips: {len(all_clips)} | Train: {len(train_clips)} | Val: {len(val_clips)}')

for split_name, clip_set in [('train', train_clips), ('val', val_clips)]:
    img_out = Path(YOLO_DIR) / split_name / 'images'
    lbl_out = Path(YOLO_DIR) / split_name / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path in Path(FRAMES_DIR).rglob('*.jpg'):
        clip_id = img_path.parent.name
        if clip_id not in clip_set: continue
        rel = img_path.relative_to(FRAMES_DIR)
        kp_path = Path(KP_DIR) / rel.with_suffix('.npy')
        if not kp_path.exists(): continue
        kp = np.load(str(kp_path))
        if kp.shape[0] == 0: continue
        kp = kp[0]
        valid = kp[kp[:, 0] > 0]
        if len(valid) < 5: continue
        x1, y1 = valid[:, 0].min(), valid[:, 1].min()
        x2, y2 = valid[:, 0].max(), valid[:, 1].max()
        cx = (x1+x2)/2; cy = (y1+y2)/2
        w = (x2-x1)*1.2; h = (y2-y1)*1.2
        cx,cy,w,h = [min(max(v,0),1) for v in [cx,cy,w,h]]
        # Visibility: 0=not labeled, 2=visible (COCO convention)
        kp_parts = []
        for i in range(17):
            x, y = kp[i, 0], kp[i, 1]
            v = 2 if (x > 0 and y > 0) else 0
            kp_parts.append(f'{x:.6f} {y:.6f} {v}')
        kp_flat = ' '.join(kp_parts)
        # class_id = 0 (person) -- single class for pose estimation
        label_line = f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {kp_flat}'
        uid = f'{clip_id}_{img_path.stem}'
        shutil.copy(str(img_path), str(img_out / f'{uid}.jpg'))
        (lbl_out / f'{uid}.txt').write_text(label_line)

# dataset.yaml: nc=1 (person) with flip_idx
yaml_content = {
    'path': YOLO_DIR, 'train': 'train/images', 'val': 'val/images',
    'nc': 1, 'names': ['person'],
    'kpt_shape': [17, 3], 'flip_idx': FLIP_IDX,
}
yaml_path = f'{YOLO_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

train_n = len(list((Path(YOLO_DIR)/'train'/'images').glob('*.jpg')))
val_n = len(list((Path(YOLO_DIR)/'val'/'images').glob('*.jpg')))
print(f'\n✅ Dataset ready: {train_n} train | {val_n} val images')
print(f'   nc=1 (person) -- pose estimation only, no exercise classification')
print(f'   flip_idx: {FLIP_IDX}')
print(f'   kpt_shape: [17, 3]')

## ✅ Step 8 — Baseline Evaluation (Pretrained YOLO26n-Pose)

In [ ]:
import json, time

baseline_model = YOLO('yolo26n-pose.pt')

# IMPORTANT: Verify labels load correctly BEFORE training
# Ref: https://docs.ultralytics.com/guides/finetuning-guide/
# "run yolo detect val ... before training to confirm labels load correctly"
print('Verifying dataset labels load correctly...')
metrics = baseline_model.val(data=yaml_path, split='val', verbose=True)

# Check for zero-label warning
train_labels = list((Path(YOLO_DIR)/'train'/'labels').glob('*.txt'))
val_labels = list((Path(YOLO_DIR)/'val'/'labels').glob('*.txt'))
print(f'\n   Train labels: {len(train_labels)}')
print(f'   Val labels:   {len(val_labels)}')
if len(train_labels) == 0 or len(val_labels) == 0:
    print('\n❌ STOP: Zero labels found! Check Step 7 — paths or exercise matching may be wrong.')
    print('   Common fix: verify EXERCISE_MAP keys match your Kaggle folder names.')
else:
    print('   ✅ Labels verified — safe to proceed to training')

# Spot-check a label file
sample_label = train_labels[0] if train_labels else None
if sample_label:
    content = sample_label.read_text().strip().split()
    n_fields = len(content)
    expected = 1 + 4 + 17*3  # class + bbox(4) + keypoints(17*3)
    print(f'\n   Sample label: {sample_label.name}')
    print(f'   Fields: {n_fields} (expected {expected})')
    if n_fields != expected:
        print(f'   ⚠️  Field count mismatch! Check label format.')
    else:
        print(f'   ✅ Label format correct')

baseline_results = {
    'model': 'yolo26n-pose-pretrained',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'pose_map50': float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0.0,
    'pose_map': float(metrics.pose.map) if hasattr(metrics, 'pose') else 0.0,
}

os.makedirs(f'{DRIVE_ROOT}/data/eval', exist_ok=True)
with open(f'{DRIVE_ROOT}/data/eval/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print(f"\n✅ Baseline OKS-mAP@0.50: {baseline_results['pose_map50']:.4f}")
print(f"   Thesis target after finetuning: >= 0.75")

## ✅ Step 9 — Two-Stage Finetune YOLO26n-Pose 🚀

> **Stage 1:** `freeze=10` — backbone frozen, head & neck learn gym exercises (20 epochs)
>
> **Stage 2:** `freeze=None` — full model fine-tune with lower LR (30 epochs)
>
> **If your Kaggle session restarts:** just re-run Steps 1–3, then jump straight to this cell.
> The resume logic detects `last.pt` on disk and picks up from the last saved epoch automatically.

In [ ]:
from ultralytics import YOLO
import os, torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# ═══════════════════════════════════════════════════════════
# STAGE 1 — Freeze backbone, train head & neck
# Auto-resumes if session restarted mid-training
# ═══════════════════════════════════════════════════════════
stage1_last = f'{MODELS_DIR}/runs/posecoach_stage1/weights/last.pt'
stage1_best = f'{MODELS_DIR}/runs/posecoach_stage1/weights/best.pt'

if os.path.exists(stage1_last):
    print(f'🔄 RESUMING Stage 1 from checkpoint: {stage1_last}')
    model = YOLO(stage1_last)
    model.train(resume=True)
else:
    print('🔒 STAGE 1: Frozen backbone — fresh start')
    model = YOLO('yolo26n-pose.pt')
    model.train(
        data=yaml_path,
        epochs=20,
        imgsz=640,
        batch=8,
        device=DEVICE,
        project=f'{MODELS_DIR}/runs',
        name='posecoach_stage1',
        exist_ok=True,
        freeze=10,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=3,
        cos_lr=True,
        patience=15,
        mosaic=0.5,
        close_mosaic=10,
        mixup=0.0,
        copy_paste=0.0,
        pose=12.0,
        save=True,
        save_period=1,
        seed=42,
        verbose=True,
    )

print('\n✅ Stage 1 complete!')
if os.path.exists(stage1_best):
    print(f'   Best: {stage1_best} ({os.path.getsize(stage1_best)/1e6:.1f} MB)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STAGE 2 — Unfreeze all layers, fine-tune full model
# Auto-resumes if session restarted mid-training
# ═══════════════════════════════════════════════════════════
stage1_best = f'{MODELS_DIR}/runs/posecoach_stage1/weights/best.pt'
stage2_last = f'{MODELS_DIR}/runs/posecoach_v1/weights/last.pt'
stage2_best = f'{MODELS_DIR}/runs/posecoach_v1/weights/best.pt'

if os.path.exists(stage2_last):
    print(f'🔄 RESUMING Stage 2 from checkpoint: {stage2_last}')
    model2 = YOLO(stage2_last)
    model2.train(resume=True)

elif os.path.exists(stage1_best):
    print('🔓 STAGE 2: Full model fine-tune — fresh start')
    print(f'   Loading Stage 1 best: {stage1_best}')
    model2 = YOLO(stage1_best)
    model2.train(
        data=yaml_path,
        epochs=30,
        imgsz=640,
        batch=8,
        device=DEVICE,
        project=f'{MODELS_DIR}/runs',
        name='posecoach_v1',
        exist_ok=True,
        freeze=None,
        optimizer='AdamW',
        lr0=0.001,
        lrf=0.01,
        warmup_epochs=1,
        cos_lr=True,
        patience=15,
        mosaic=0.5,
        close_mosaic=10,
        mixup=0.0,
        copy_paste=0.0,
        pose=12.0,
        save=True,
        save_period=1,
        seed=42,
        verbose=True,
    )
else:
    print(f'❌ Stage 1 weights not found at: {stage1_best}')
    raise FileNotFoundError('Run Stage 1 cell first.')

print('\n✅ Two-stage training complete!')
if os.path.exists(stage2_best):
    print(f'   Best: {stage2_best} ({os.path.getsize(stage2_best)/1e6:.1f} MB)')

## ✅ Step 10 — Save Final Weights & Evaluate

In [ ]:
import shutil

best_weights = f'{MODELS_DIR}/runs/posecoach_v1/weights/best.pt'
final_path = f'{MODELS_DIR}/yolo_posecoach_v1.pt'

if os.path.exists(best_weights):
    shutil.copy(best_weights, final_path)
    size_mb = os.path.getsize(final_path) / 1e6
    print(f'✅ Weights saved: {final_path} ({size_mb:.1f} MB)')
else:
    print('❌ best.pt not found — check training logs above')

finetuned = YOLO(final_path)
metrics = finetuned.val(data=yaml_path, split='val', verbose=False)

final_results = {
    'model': 'yolo26n-pose-posecoach-v1',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'weights_path': final_path,
    'pose_map50': float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0.0,
    'pose_map': float(metrics.pose.map) if hasattr(metrics, 'pose') else 0.0,
    'thesis_gate_passed': (float(metrics.pose.map50) if hasattr(metrics, 'pose') else 0) >= 0.75
}

with open(f'{DRIVE_ROOT}/data/eval/yolo_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"\n📊 Final OKS-mAP@0.50: {final_results['pose_map50']:.4f}")
print(f"   Thesis gate (>= 0.75): {'✅ PASSED' if final_results['thesis_gate_passed'] else '❌ FAILED — train more epochs'}")
print(f"\n=== NEXT STEPS ===")
print(f'1. Download weights from Drive: {final_path}')
print('2. Place in local: posecoach/models/yolo_posecoach_v1.pt')
print('3. Update .env: MODEL_PATH=models/yolo_posecoach_v1.pt')
print('4. Proceed to P02 — Infrastructure')

## Step 10b -- ONNX Export and CPU Latency Benchmark

> **Thesis requirement:** sub-80ms end-to-end inference on CPU at 15 FPS.
> YOLO26n detection runs ~39ms on CPU via ONNX. Pose adds keypoint head.
> We must measure actual latency BEFORE building P02.
>
> Budget: preprocessing ~5ms + YOLO ~??ms + angles ~2ms + WebSocket ~10ms = <80ms


In [ ]:
# ── Export to ONNX for production CPU inference ──
finetuned = YOLO(f'{MODELS_DIR}/yolo_posecoach_v1.pt')

onnx_path = finetuned.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=17,
    dynamic=False,        # static shape = fastest inference
)
print(f'✅ ONNX model exported: {onnx_path}')
onnx_size = os.path.getsize(onnx_path) / 1e6
print(f'   Size: {onnx_size:.1f} MB')

# ── CPU Latency Benchmark ──
import time
import numpy as np

onnx_model = YOLO(onnx_path)

# Realistic test image
test_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(5):
    onnx_model.predict(test_img, verbose=False, device='cpu')

# Benchmark: 50 iterations on CPU
latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    results = onnx_model.predict(test_img, verbose=False, device='cpu')
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)

latencies = np.array(latencies)
print(f'\n📊 CPU Inference Latency (ONNX, 50 runs):')
print(f'   Mean:   {latencies.mean():.1f} ms')
print(f'   Median: {np.median(latencies):.1f} ms')
print(f'   P95:    {np.percentile(latencies, 95):.1f} ms')
print(f'   P99:    {np.percentile(latencies, 99):.1f} ms')

# Budget check
PREPROCESSING_MS = 5
ANGLE_COMPUTE_MS = 2
WEBSOCKET_MS = 10
total_budget = latencies.mean() + PREPROCESSING_MS + ANGLE_COMPUTE_MS + WEBSOCKET_MS

print(f'\n🎯 End-to-End Latency Estimate:')
print(f'   YOLO inference:  {latencies.mean():.1f} ms')
print(f'   + Preprocessing: {PREPROCESSING_MS} ms')
print(f'   + Angle compute: {ANGLE_COMPUTE_MS} ms')
print(f'   + WebSocket:     {WEBSOCKET_MS} ms')
print(f'   Total estimate:  {total_budget:.1f} ms')
print(f'   Thesis target:   80 ms')
gate = '✅ PASSED' if total_budget < 80 else ('⚠️ TIGHT' if total_budget < 100 else '❌ FAILED')
print(f'   Gate: {gate}')

# Save benchmark
benchmark = {
    'model': 'yolo26n-pose-posecoach-v1-onnx',
    'format': 'ONNX', 'device': 'CPU (Colab)', 'imgsz': 640,
    'mean_ms': round(float(latencies.mean()), 1),
    'median_ms': round(float(np.median(latencies)), 1),
    'p95_ms': round(float(np.percentile(latencies, 95)), 1),
    'estimated_total_ms': round(float(total_budget), 1),
    'thesis_gate_80ms': total_budget < 80
}
with open(f'{DRIVE_ROOT}/data/eval/latency_benchmark.json', 'w') as f:
    json.dump(benchmark, f, indent=2)

import shutil
onnx_final = f'{MODELS_DIR}/yolo_posecoach_v1.onnx'
shutil.copy(onnx_path, onnx_final)
print(f'\n💾 ONNX saved: {onnx_final}')


## Step 10c -- Joint Angle MAE Evaluation (Thesis Metric)

> **Thesis requirement:** MAE <=5 degrees for joint angles.
> OKS-mAP measures keypoint spatial accuracy, NOT angle accuracy.
> This step computes actual 2D joint angles and measures MAE.


In [ ]:
# ── Joint Angle MAE Evaluation (Thesis Metric: MAE <= 5 degrees) ──

import numpy as np
from pathlib import Path

def compute_angle_2d(a, b, c):
    """Angle at point b in degrees (2D)."""
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

# COCO 17-keypoint indices
ANGLE_JOINTS_2D = {
    'left_knee':     (11, 13, 15),  # hip -> knee -> ankle
    'right_knee':    (12, 14, 16),
    'left_hip':      (5, 11, 13),   # shoulder -> hip -> knee
    'right_hip':     (6, 12, 14),
    'left_elbow':    (5, 7, 9),     # shoulder -> elbow -> wrist
    'right_elbow':   (6, 8, 10),
    'left_shoulder': (7, 5, 11),    # elbow -> shoulder -> hip
    'right_shoulder':(8, 6, 12),
}

finetuned = YOLO(f'{MODELS_DIR}/yolo_posecoach_v1.pt')
val_images = sorted((Path(YOLO_DIR) / 'val' / 'images').glob('*.jpg'))
val_labels = Path(YOLO_DIR) / 'val' / 'labels'

angle_errors = {name: [] for name in ANGLE_JOINTS_2D}
processed = skipped = 0

for img_path in tqdm(val_images, desc='Computing angle MAE'):
    label_file = val_labels / f'{img_path.stem}.txt'
    if not label_file.exists():
        skipped += 1
        continue

    content = label_file.read_text().strip().split()
    if len(content) != 1 + 4 + 17*3:
        skipped += 1
        continue

    gt_kp = []
    for i in range(17):
        base = 5 + i*3
        x, y, v = float(content[base]), float(content[base+1]), int(content[base+2])
        gt_kp.append((x, y, v))

    results = finetuned.predict(str(img_path), verbose=False, conf=0.3, device=DEVICE)
    if results[0].keypoints is None or len(results[0].keypoints) == 0:
        skipped += 1
        continue

    pred_kp_raw = results[0].keypoints.xyn.cpu().numpy()
    if pred_kp_raw.shape[0] == 0:
        skipped += 1
        continue
    pred_kp = pred_kp_raw[0]

    for angle_name, (a_idx, b_idx, c_idx) in ANGLE_JOINTS_2D.items():
        gt_vis = all(gt_kp[j][2] > 0 for j in [a_idx, b_idx, c_idx])
        pred_vis = all(pred_kp[j][0] > 0 and pred_kp[j][1] > 0 for j in [a_idx, b_idx, c_idx])

        if gt_vis and pred_vis:
            gt_angle = compute_angle_2d(
                (gt_kp[a_idx][0], gt_kp[a_idx][1]),
                (gt_kp[b_idx][0], gt_kp[b_idx][1]),
                (gt_kp[c_idx][0], gt_kp[c_idx][1])
            )
            pred_angle = compute_angle_2d(
                (pred_kp[a_idx][0], pred_kp[a_idx][1]),
                (pred_kp[b_idx][0], pred_kp[b_idx][1]),
                (pred_kp[c_idx][0], pred_kp[c_idx][1])
            )
            angle_errors[angle_name].append(abs(pred_angle - gt_angle))

    processed += 1

# Report
print(f'\n📊 Joint Angle MAE ({processed} images, {skipped} skipped)')
print(f'{"="*55}')

all_errors = []
for name, errors in angle_errors.items():
    if errors:
        mae = np.mean(errors)
        std = np.std(errors)
        p95 = np.percentile(errors, 95)
        status = '✅' if mae <= 5.0 else ('⚠️' if mae <= 8.0 else '❌')
        print(f'  {status} {name:20s}: MAE={mae:5.2f} +/- {std:5.2f} (P95={p95:5.2f}, n={len(errors)})')
        all_errors.extend(errors)

if all_errors:
    overall_mae = np.mean(all_errors)
    print(f'\n  Overall MAE: {overall_mae:.2f} degrees')
    gate = '✅ PASSED' if overall_mae <= 5.0 else '❌ FAILED'
    print(f'  Thesis gate (<=5 deg): {gate}')

    mae_results = {
        'model': 'yolo26n-pose-posecoach-v1',
        'overall_mae_degrees': round(float(overall_mae), 2),
        'thesis_gate_5deg': overall_mae <= 5.0,
        'per_joint': {
            name: {
                'mae': round(float(np.mean(errs)), 2),
                'std': round(float(np.std(errs)), 2),
                'p95': round(float(np.percentile(errs, 95)), 2),
                'n_samples': len(errs)
            }
            for name, errs in angle_errors.items() if errs
        }
    }
    with open(f'{DRIVE_ROOT}/data/eval/angle_mae_results.json', 'w') as f:
        json.dump(mae_results, f, indent=2)
    print(f'\n  Saved to: {DRIVE_ROOT}/data/eval/angle_mae_results.json')

    print(f'\n  Note: This MAE is against pseudo-GT (pretrained YOLO predictions).')
    print(f'  For true clinical accuracy, compare against Fit3D projected angles in Step 11c.')


## ✅ Step 11 — Fit3D Dataset: Download & Build Golden Angle Templates

> **Why Fit3D?** Vicon motion-capture gives sub-degree-accurate 3D joint positions (25 joints @ 50 FPS).
> This lets us compute biomechanically precise angle ranges for each exercise — the ground truth for `form_scorer.py`.
>
> The Kaggle pipeline (Steps 1–10) trains the **pose model**. Fit3D calibrates the **form scoring**.
> These two pipelines are independent and complementary.

### What this step does:
1. Downloads Fit3D to Google Drive (not local!)
2. Explores the dataset structure
3. Computes joint angle distributions per exercise
4. Exports golden `ANGLE_RANGES` as JSON for your form scorer


### 11a — Download Fit3D to Google Drive

> **You need to download Fit3D manually** because it requires authenticated access.
>
> **Option A (Recommended):** Download via browser → upload to Drive
> 1. Go to [fit3d.imar.ro/download](https://fit3d.imar.ro/download)
> 2. Download "Dataset Info" (12KB) and "Training Set" (18GB)
> 3. Upload both to your Google Drive under `GYMVISION AI/datasets/fit3d/`
>
> **Option B:** If Fit3D provides direct download links, paste them below


In [ ]:
# ── Fit3D directory setup ──
# Attach Fit3D via the Data tab (right panel → Add Data → upload your Fit3D zip)
# OR manually place files in /kaggle/working/gymvision/datasets/fit3d/raw/
FIT3D_DIR           = f'{KAGGLE_ROOT}/datasets/fit3d'
FIT3D_RAW           = f'{FIT3D_DIR}/raw'
FIT3D_PROCESSED     = f'{FIT3D_DIR}/processed'
ANGLE_TEMPLATES_DIR = f'{FIT3D_DIR}/angle_templates'

for d in [FIT3D_DIR, FIT3D_RAW, FIT3D_PROCESSED, ANGLE_TEMPLATES_DIR]:
    os.makedirs(d, exist_ok=True)

# Check if Fit3D was attached via Data tab
fit3d_input = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any('fit3d' in f.lower() or 'joints' in f.lower() for f in files):
        fit3d_input = root
        break

if fit3d_input:
    print(f'✅ Fit3D data found at {fit3d_input} — symlinking to {FIT3D_RAW}')
    for item in os.listdir(fit3d_input):
        src_p = os.path.join(fit3d_input, item)
        dst_p = os.path.join(FIT3D_RAW, item)
        if not os.path.exists(dst_p):
            os.symlink(src_p, dst_p)
else:
    fit3d_files = [f for r,d,files in os.walk(FIT3D_RAW) for f in files]
    if fit3d_files:
        print(f'✅ Fit3D already in working dir: {len(fit3d_files)} files')
    else:
        print('⚠️  Fit3D not found.')
        print(f'   Upload to: {FIT3D_RAW}')
        print('   Or attach via Data tab → Add Data')

# Show structure
for item in os.listdir(FIT3D_RAW)[:20]:
    item_path = os.path.join(FIT3D_RAW, item)
    if os.path.isdir(item_path):
        print(f'  📁 {item}/ ({len(os.listdir(item_path))} items)')
    else:
        print(f'  📄 {item} ({os.path.getsize(item_path)/1e6:.1f} MB)')

### 11b — Explore Fit3D Dataset Structure

Run this after uploading Fit3D to Drive. It maps out the exercises, subjects, and data formats available.


In [ ]:
import json
from collections import defaultdict
from pathlib import Path

# ── Auto-detect Fit3D structure ──
fit3d_root = Path(FIT3D_RAW)

# Find all directories that might contain 3D joint data
joint_files = list(fit3d_root.rglob('*joints*'))
smplx_files = list(fit3d_root.rglob('*smplx*'))
npz_files = list(fit3d_root.rglob('*.npz'))
json_files = list(fit3d_root.rglob('*.json'))
csv_files = list(fit3d_root.rglob('*.csv'))

print('=== Fit3D Data Discovery ===')
print(f'  Joint files:  {len(joint_files)}')
print(f'  SMPLX files:  {len(smplx_files)}')
print(f'  NPZ files:    {len(npz_files)}')
print(f'  JSON files:   {len(json_files)}')
print(f'  CSV files:    {len(csv_files)}')

# Map exercises found
exercises_found = defaultdict(int)
subjects_found = set()

for p in fit3d_root.rglob('*'):
    if p.is_dir():
        parts = p.relative_to(fit3d_root).parts
        if len(parts) >= 2:
            # Typically: train/s01/exercise_name or similar
            if parts[0] in ('train', 'test') and len(parts) >= 3:
                subjects_found.add(parts[1])
                exercises_found[parts[2]] += 1
            elif parts[0].startswith('s') and len(parts) >= 2:
                subjects_found.add(parts[0])
                exercises_found[parts[1]] += 1

if exercises_found:
    print(f'\n=== Exercises Found ({len(exercises_found)}) ===')
    for ex, count in sorted(exercises_found.items()):
        print(f'  {ex}: {count} sequences')
    print(f'\n=== Subjects: {len(subjects_found)} ===')
    print(f'  {sorted(subjects_found)[:10]}{"..." if len(subjects_found) > 10 else ""}')
else:
    print('\n⚠️  Could not auto-detect exercise structure.')
    print('   Showing raw directory tree (first 3 levels):')
    for p in sorted(fit3d_root.rglob('*'))[:50]:
        rel = p.relative_to(fit3d_root)
        if len(rel.parts) <= 3:
            indent = '  ' * len(rel.parts)
            suffix = '/' if p.is_dir() else f' ({p.stat().st_size/1e3:.0f}KB)'
            print(f'{indent}{p.name}{suffix}')

# Try to read Dataset Info if present
info_files = list(fit3d_root.rglob('*info*')) + list(fit3d_root.rglob('*README*'))
if info_files:
    print(f'\n=== Dataset Info Files ===')
    for f in info_files[:5]:
        print(f'  {f.relative_to(fit3d_root)}')


### 11c -- Compute Golden Angle Templates from 3D Skeletons

Key step: extract joint angles from Fit3D mocap data for ANGLE_RANGES.

**IMPORTANT: 2D vs 3D angle gap**
Fit3D gives 3D joints, but runtime uses 2D keypoints. We compute angles both ways:
1. **3D angles** -- true biomechanical (thesis reference)
2. **2D projected angles** -- what your system sees (for production ANGLE_RANGES)

We use orthographic projections (front/side/45 deg) to approximate camera viewpoints.

In [ ]:
import numpy as np
from pathlib import Path
import json
from collections import defaultdict

# ── Joint angle computation utilities ──

def compute_angle_3d(a, b, c):
    """Compute angle at joint b formed by segments a→b and b→c (in degrees).
    a, b, c are 3D points as numpy arrays."""
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

# ── Fit3D joint indices (adjust if your download uses different ordering) ──
# Common 25-joint Fit3D skeleton (verify with your Dataset Info):
# This mapping may need adjustment based on actual Fit3D format
FIT3D_JOINTS = {
    'pelvis': 0, 'left_hip': 1, 'right_hip': 2,
    'spine1': 3, 'left_knee': 4, 'right_knee': 5,
    'spine2': 6, 'left_ankle': 7, 'right_ankle': 8,
    'spine3': 9, 'left_foot': 10, 'right_foot': 11,
    'neck': 12, 'left_collar': 13, 'right_collar': 14,
    'head': 15, 'left_shoulder': 16, 'right_shoulder': 17,
    'left_elbow': 18, 'right_elbow': 19,
    'left_wrist': 20, 'right_wrist': 21,
    'left_hand': 22, 'right_hand': 23, 'jaw': 24
}

# ── Angle definitions: (parent, joint, child) triplets ──
ANGLE_DEFS = {
    'left_knee_flexion':   ('left_hip', 'left_knee', 'left_ankle'),
    'right_knee_flexion':  ('right_hip', 'right_knee', 'right_ankle'),
    'left_hip_flexion':    ('left_shoulder', 'left_hip', 'left_knee'),
    'right_hip_flexion':   ('right_shoulder', 'right_hip', 'right_knee'),
    'left_elbow_flexion':  ('left_shoulder', 'left_elbow', 'left_wrist'),
    'right_elbow_flexion': ('right_shoulder', 'right_elbow', 'right_wrist'),
    'torso_lean':          ('neck', 'pelvis', 'left_ankle'),  # proxy for forward lean
}

def extract_angles_from_sequence(joints_3d):
    """Given a (T, 25, 3) array of 3D joints, compute angles per frame.
    Returns dict of {angle_name: array of shape (T,)}"""
    T = joints_3d.shape[0]
    angles = {}
    for angle_name, (p, j, c) in ANGLE_DEFS.items():
        p_idx = FIT3D_JOINTS.get(p)
        j_idx = FIT3D_JOINTS.get(j)
        c_idx = FIT3D_JOINTS.get(c)
        if p_idx is None or j_idx is None or c_idx is None:
            continue
        frame_angles = np.array([
            compute_angle_3d(joints_3d[t, p_idx], joints_3d[t, j_idx], joints_3d[t, c_idx])
            for t in range(T)
        ])
        angles[angle_name] = frame_angles
    return angles

print('✅ Angle computation utilities loaded')
print(f'   Defined {len(ANGLE_DEFS)} angle measurements')
for name, (p, j, c) in ANGLE_DEFS.items():
    print(f'   • {name}: {p} → {j} → {c}')


In [ ]:
# ── Load 3D joints and compute angle templates per exercise ──

fit3d_root = Path(FIT3D_RAW)
angle_stats = defaultdict(lambda: defaultdict(list))  # {exercise: {angle: [values]}}

# Find all 3D joint files (common formats: .npz, .npy, .json, .csv)
joint_paths = (
    list(fit3d_root.rglob('*joints3d*.npz')) +
    list(fit3d_root.rglob('*joints3d*.npy')) +
    list(fit3d_root.rglob('*joint_positions*.npz')) +
    list(fit3d_root.rglob('*joints*.npz'))
)

if not joint_paths:
    # Fallback: try all .npz files
    joint_paths = list(fit3d_root.rglob('*.npz'))
    print(f'⚠️  No files matching *joints3d* found. Trying all .npz files ({len(joint_paths)} found)')

print(f'Found {len(joint_paths)} potential joint files')

# Process each file
processed = skipped = 0
for jp in tqdm(joint_paths, desc='Processing 3D joints'):
    try:
        # Determine exercise name from path
        rel_parts = jp.relative_to(fit3d_root).parts
        exercise_name = None
        for part in rel_parts:
            # Skip subject folders (s01, s02, ...) and split folders (train/test)
            if part in ('train', 'test') or part.startswith('s') and len(part) <= 3:
                continue
            if part.endswith('.npz') or part.endswith('.npy'):
                continue
            exercise_name = part
            break

        if not exercise_name:
            exercise_name = jp.stem

        # Load 3D joints
        if jp.suffix == '.npz':
            data = np.load(str(jp), allow_pickle=True)
            # Try common key names
            joints = None
            for key in ['joints3d', 'joint_positions', 'poses', 'keypoints3d', 'body']:
                if key in data:
                    joints = data[key]
                    break
            if joints is None:
                # Use first array in the file
                keys = list(data.keys())
                if keys:
                    joints = data[keys[0]]
        elif jp.suffix == '.npy':
            joints = np.load(str(jp), allow_pickle=True)
        else:
            skipped += 1
            continue

        if joints is None or joints.ndim < 2:
            skipped += 1
            continue

        # Reshape if needed: expect (T, J, 3)
        if joints.ndim == 2 and joints.shape[1] % 3 == 0:
            n_joints = joints.shape[1] // 3
            joints = joints.reshape(-1, n_joints, 3)
        elif joints.ndim != 3:
            skipped += 1
            continue

        # Only process if we have ~25 joints
        if joints.shape[1] < 20:
            skipped += 1
            continue

        # Compute angles for this sequence
        angles = extract_angles_from_sequence(joints)
        for angle_name, values in angles.items():
            angle_stats[exercise_name][angle_name].extend(values.tolist())

        processed += 1

    except Exception as e:
        skipped += 1
        continue

print(f'\n✅ Processed: {processed} | Skipped: {skipped}')
print(f'   Exercises with angle data: {len(angle_stats)}')
for ex in sorted(angle_stats.keys()):
    n_angles = len(angle_stats[ex])
    n_frames = len(next(iter(angle_stats[ex].values()))) if angle_stats[ex] else 0
    print(f'   • {ex}: {n_angles} angles, ~{n_frames:,} frames')


In [ ]:
# ── Build golden ANGLE_RANGES: 3D (reference) + 2D projected (production) ──
# Why both? Fit3D gives 3D mocap joints, but runtime uses 2D keypoints from a
# single camera. 3D angles != 2D angles due to perspective/depth. We compute
# orthographic 2D projections from multiple viewpoints to approximate the range
# of 2D angles a monocular camera would see.

import numpy as np

def compute_angle_2d_pts(a, b, c):
    """Angle at point b in degrees, from 2D points."""
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0)))

# ── Orthographic projection matrices (drop one axis) ──
# Front view  = drop Z (project onto XY)
# Side view   = drop X (project onto YZ)
# 45° view    = rotate 45° around Y then drop Z
cos45, sin45 = np.cos(np.pi/4), np.sin(np.pi/4)
PROJECTIONS = {
    'front': lambda pts: pts[:, :2],           # (X, Y)
    'side':  lambda pts: pts[:, 1:3],          # (Y, Z)
    '45deg': lambda pts: np.column_stack([     # rotated X, Y
        pts[:, 0] * cos45 + pts[:, 2] * sin45,
        pts[:, 1]
    ]),
}

# ═══ 3D angle ranges ═══
ANGLE_RANGES_3D = {}
for exercise, angles in sorted(angle_stats.items()):
    ANGLE_RANGES_3D[exercise] = {}
    for angle_name, values in angles.items():
        arr = np.array(values)
        arr = arr[~np.isnan(arr)]
        if len(arr) < 10:
            continue
        mean = float(np.mean(arr))
        std = float(np.std(arr))
        ANGLE_RANGES_3D[exercise][angle_name] = {
            'min': round(max(0, mean - 2*std), 1),
            'max': round(min(180, mean + 2*std), 1),
            'mean': round(mean, 1),
            'std': round(std, 1),
            'n_samples': len(arr),
        }

# ═══ 2D projected angle ranges ═══
# Re-process the 3D joint files to compute 2D projected angles
fit3d_root = Path(FIT3D_RAW)
angle_stats_2d = defaultdict(lambda: defaultdict(list))

joint_paths = (
    list(fit3d_root.rglob('*joints3d*.npz')) +
    list(fit3d_root.rglob('*joints3d*.npy')) +
    list(fit3d_root.rglob('*joint_positions*.npz')) +
    list(fit3d_root.rglob('*joints*.npz'))
)
if not joint_paths:
    joint_paths = list(fit3d_root.rglob('*.npz'))

processed_2d = 0
for jp in tqdm(joint_paths, desc='Computing 2D projected angles'):
    try:
        rel_parts = jp.relative_to(fit3d_root).parts
        exercise_name = None
        for part in rel_parts:
            if part in ('train', 'test') or (part.startswith('s') and len(part) <= 3):
                continue
            if part.endswith('.npz') or part.endswith('.npy'):
                continue
            exercise_name = part
            break
        if not exercise_name:
            exercise_name = jp.stem

        if jp.suffix == '.npz':
            data = np.load(str(jp), allow_pickle=True)
            joints = None
            for key in ['joints3d', 'joint_positions', 'poses', 'keypoints3d', 'body']:
                if key in data:
                    joints = data[key]
                    break
            if joints is None:
                keys = list(data.keys())
                if keys:
                    joints = data[keys[0]]
        elif jp.suffix == '.npy':
            joints = np.load(str(jp), allow_pickle=True)
        else:
            continue

        if joints is None or joints.ndim < 2:
            continue

        if joints.ndim == 2 and joints.shape[1] % 3 == 0:
            joints = joints.reshape(-1, joints.shape[1] // 3, 3)
        elif joints.ndim != 3:
            continue

        if joints.shape[1] < 20:
            continue

        T = joints.shape[0]
        # For each projection, compute angles
        for proj_name, proj_fn in PROJECTIONS.items():
            for t in range(T):
                pts_2d = proj_fn(joints[t])  # (J, 2)
                for angle_name, (p, j, c) in ANGLE_DEFS.items():
                    p_idx = FIT3D_JOINTS.get(p)
                    j_idx = FIT3D_JOINTS.get(j)
                    c_idx = FIT3D_JOINTS.get(c)
                    if p_idx is None or j_idx is None or c_idx is None:
                        continue
                    a2d = compute_angle_2d_pts(pts_2d[p_idx], pts_2d[j_idx], pts_2d[c_idx])
                    angle_stats_2d[exercise_name][angle_name].append(a2d)
        processed_2d += 1
    except Exception:
        continue

print(f'\n✅ 2D projected angles computed from {processed_2d} sequences across {len(PROJECTIONS)} viewpoints')

ANGLE_RANGES_2D = {}
for exercise, angles in sorted(angle_stats_2d.items()):
    ANGLE_RANGES_2D[exercise] = {}
    for angle_name, values in angles.items():
        arr = np.array(values)
        arr = arr[~np.isnan(arr)]
        if len(arr) < 10:
            continue
        mean = float(np.mean(arr))
        std = float(np.std(arr))
        ANGLE_RANGES_2D[exercise][angle_name] = {
            'min': round(max(0, mean - 2*std), 1),
            'max': round(min(180, mean + 2*std), 1),
            'mean': round(mean, 1),
            'std': round(std, 1),
            'n_samples': len(arr),
            'note': 'Aggregated over front/side/45deg orthographic projections',
        }

# ═══ Save both versions ═══
path_3d = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges_3d.json'
path_2d = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges_2d.json'

with open(path_3d, 'w') as f:
    json.dump(ANGLE_RANGES_3D, f, indent=2)
with open(path_2d, 'w') as f:
    json.dump(ANGLE_RANGES_2D, f, indent=2)

# Compact version for form_scorer.py — USE 2D (what runtime actually sees)
compact_ranges = {}
for exercise, angles in ANGLE_RANGES_2D.items():
    compact_ranges[exercise] = {
        name: [data['min'], data['max']]
        for name, data in angles.items()
    }
compact_path = f'{ANGLE_TEMPLATES_DIR}/angle_ranges_compact.json'
with open(compact_path, 'w') as f:
    json.dump(compact_ranges, f, indent=2)

print(f'\n✅ Golden Angle Ranges exported!')
print(f'   3D (thesis reference):      {path_3d}')
print(f'   2D projected (production):  {path_2d}')
print(f'   Compact (for form_scorer):  {compact_path}')

# Pretty-print comparison
print('\n=== 3D vs 2D Angle Comparison (first exercise) ===')
for exercise in list(set(ANGLE_RANGES_3D.keys()) & set(ANGLE_RANGES_2D.keys()))[:2]:
    print(f'\n  📐 {exercise}:')
    for angle in ANGLE_RANGES_3D[exercise]:
        r3 = ANGLE_RANGES_3D[exercise][angle]
        r2 = ANGLE_RANGES_2D[exercise].get(angle, {})
        if r2:
            gap = abs(r3['mean'] - r2['mean'])
            print(f'     {angle}:')
            print(f'       3D: {r3["min"]}°–{r3["max"]}° (μ={r3["mean"]}°)')
            print(f'       2D: {r2["min"]}°–{r2["max"]}° (μ={r2["mean"]}°)')
            print(f'       Gap: {gap:.1f}° — {"⚠️ significant" if gap > 5 else "✅ small"}')

# Exercise mapping suggestions
print('\n=== Exercise Mapping Suggestions ===')
your_exercises = ['squat', 'deadlift', 'curl', 'bench', 'ohp', 'lunge', 'plank']
for your_ex in your_exercises:
    matches = [ex for ex in ANGLE_RANGES_2D.keys() if your_ex in ex.lower()]
    if matches:
        print(f'  ✅ {your_ex} → {matches}')
    else:
        print(f'  ❓ {your_ex} → no exact match in Fit3D')

## ✅ Step 12 — Validate Rep Counter with Fit3D Ground Truth

> Fit3D provides **repetition time intervals** (≥5 reps per sequence).
> We use these to measure how accurately our `rep_counter.py` detects reps.
>
> **Metric:** Rep count accuracy = |predicted_reps - gt_reps| / gt_reps


In [ ]:
# ── Load Fit3D rep annotations ──
# Fit3D stores rep segmentation info (format varies — check Dataset Info)

fit3d_root = Path(FIT3D_RAW)

# Search for rep/segment annotation files
rep_files = (
    list(fit3d_root.rglob('*rep*')) +
    list(fit3d_root.rglob('*segment*')) +
    list(fit3d_root.rglob('*annotation*')) +
    list(fit3d_root.rglob('*action*'))
)

print(f'Found {len(rep_files)} potential rep/segment annotation files')
for rf in rep_files[:20]:
    size = rf.stat().st_size / 1e3 if rf.is_file() else 0
    print(f'  {rf.relative_to(fit3d_root)} ({size:.1f}KB)')

# ── Try to load rep annotations ──
rep_ground_truth = {}  # {sequence_id: {'exercise': str, 'n_reps': int, 'intervals': list}}

for rf in rep_files:
    try:
        if rf.suffix == '.json':
            with open(rf) as f:
                data = json.load(f)
            # Parse based on structure
            if isinstance(data, dict):
                for key in ['reps', 'repetitions', 'segments', 'annotations']:
                    if key in data:
                        seq_id = rf.stem
                        rep_ground_truth[seq_id] = {
                            'exercise': rf.parent.name,
                            'data': data[key],
                            'n_reps': len(data[key]) if isinstance(data[key], list) else data[key]
                        }
                        break
        elif rf.suffix == '.csv':
            import csv
            with open(rf) as f:
                reader = csv.reader(f)
                rows = list(reader)
            if len(rows) > 1:
                seq_id = rf.stem
                rep_ground_truth[seq_id] = {
                    'exercise': rf.parent.name,
                    'n_reps': len(rows) - 1,  # subtract header
                    'raw_rows': rows
                }
        elif rf.suffix in ('.npz', '.npy'):
            data = np.load(str(rf), allow_pickle=True)
            if isinstance(data, np.ndarray) and data.ndim >= 1:
                seq_id = rf.stem
                rep_ground_truth[seq_id] = {
                    'exercise': rf.parent.name,
                    'n_reps': len(data),
                    'data': data.tolist() if hasattr(data, 'tolist') else str(data)
                }
    except Exception as e:
        continue

print(f'\n✅ Loaded rep ground truth for {len(rep_ground_truth)} sequences')
for seq_id, info in list(rep_ground_truth.items())[:10]:
    print(f'  {seq_id}: {info["exercise"]} — {info.get("n_reps", "?")} reps')


In [ ]:
# ── Rep counting validation using peak detection ──
# This simulates what rep_counter.py does: detect angle oscillation peaks

from scipy.signal import find_peaks

def count_reps_from_angles(angle_sequence, prominence=10, distance=15):
    """Count reps by detecting peaks in a joint angle time series.
    prominence: minimum angle change to count as a rep (degrees)
    distance: minimum frames between reps (at 50fps, 15 frames = 0.3s)
    """
    if len(angle_sequence) < 10:
        return 0, []
    peaks, properties = find_peaks(angle_sequence, prominence=prominence, distance=distance)
    return len(peaks), peaks.tolist()

# ── Validate against Fit3D ground truth ──
validation_results = []

for seq_id, gt_info in rep_ground_truth.items():
    exercise = gt_info['exercise']
    gt_reps = gt_info.get('n_reps', 0)
    if gt_reps == 0 or gt_reps is None:
        continue

    # Find corresponding 3D joint file
    possible_paths = list(fit3d_root.rglob(f'*{seq_id}*joints*')) + list(fit3d_root.rglob(f'*{seq_id}*.npz'))
    if not possible_paths:
        continue

    try:
        jp = possible_paths[0]
        if jp.suffix == '.npz':
            data = np.load(str(jp), allow_pickle=True)
            joints = None
            for key in ['joints3d', 'joint_positions', 'poses', 'keypoints3d']:
                if key in data:
                    joints = data[key]
                    break
            if joints is None:
                joints = data[list(data.keys())[0]]
        else:
            joints = np.load(str(jp), allow_pickle=True)

        if joints.ndim == 2:
            joints = joints.reshape(-1, joints.shape[1] // 3, 3)

        # Use knee flexion as primary rep signal (works for squats, lunges, etc.)
        angles = extract_angles_from_sequence(joints)
        # Pick the angle with most variation (likely the primary movement)
        best_angle = None
        best_std = 0
        for name, vals in angles.items():
            s = np.std(vals)
            if s > best_std:
                best_std = s
                best_angle = name

        if best_angle:
            pred_reps, peaks = count_reps_from_angles(
                np.array(angles[best_angle]),
                prominence=best_std * 0.5,
                distance=25  # 0.5s at 50fps
            )
            error = abs(pred_reps - gt_reps)
            validation_results.append({
                'seq_id': seq_id,
                'exercise': exercise,
                'gt_reps': gt_reps,
                'pred_reps': pred_reps,
                'error': error,
                'angle_used': best_angle
            })
    except Exception:
        continue

if validation_results:
    print(f'=== Rep Counter Validation ({len(validation_results)} sequences) ===\n')
    total_gt = sum(r['gt_reps'] for r in validation_results)
    total_pred = sum(r['pred_reps'] for r in validation_results)
    total_error = sum(r['error'] for r in validation_results)
    accuracy = 1 - (total_error / total_gt) if total_gt > 0 else 0

    for r in validation_results[:15]:
        status = '✅' if r['error'] == 0 else ('⚠️' if r['error'] <= 1 else '❌')
        print(f'  {status} {r["exercise"]}/{r["seq_id"]}: GT={r["gt_reps"]}, Pred={r["pred_reps"]} (Δ={r["error"]}) [{r["angle_used"]}]')

    print(f'\n  📊 Overall: {total_pred}/{total_gt} reps detected')
    print(f'     Accuracy: {accuracy:.1%}')
    print(f'     Mean absolute error: {total_error/len(validation_results):.2f} reps')

    # Save validation results
    val_path = f'{DRIVE_ROOT}/data/eval/rep_counter_validation.json'
    os.makedirs(os.path.dirname(val_path), exist_ok=True)
    with open(val_path, 'w') as f:
        json.dump({
            'results': validation_results,
            'summary': {
                'n_sequences': len(validation_results),
                'accuracy': round(accuracy, 4),
                'mean_abs_error': round(total_error/len(validation_results), 2)
            }
        }, f, indent=2)
    print(f'\n  💾 Saved to: {val_path}')
else:
    print('⚠️  No sequences could be matched for validation.')
    print('   This may mean:')
    print('   1. Rep annotations are in a different format — check the Dataset Info file')
    print('   2. File naming doesn\'t match between joints and annotations')
    print('   3. Run the exploration cell (11b) to understand the structure first')


## P01 Complete -- Summary

Dual-dataset pipeline with all thesis metrics evaluated:

| Thesis Metric | Step | Output |
|---|---|---|
| OKS-mAP >=0.75 | Step 10 | data/eval/yolo_results.json |
| MAE <=5 deg angles | Step 10c | data/eval/angle_mae_results.json |
| Sub-80ms CPU latency | Step 10b | data/eval/latency_benchmark.json |
| Rep counter accuracy | Step 12 | data/eval/rep_counter_validation.json |

### Key outputs on Google Drive:
- models/yolo_posecoach_v1.pt (PyTorch)
- models/yolo_posecoach_v1.onnx (ONNX for FastAPI)
- datasets/fit3d/angle_templates/golden_angle_ranges_3d.json
- datasets/fit3d/angle_templates/golden_angle_ranges_2d.json
- datasets/fit3d/angle_templates/angle_ranges_compact.json

### Architecture: nc=1 (person) pose model
Exercise classification is NOT in the pose model. Handle via:
- Option A: User selects exercise in app (recommended)
- Option B: Classify from keypoint sequence over time (future work)

### Next: P02
1. Download yolo_posecoach_v1.onnx from Drive
2. Copy angle_ranges_compact.json into project
3. If latency >80ms, consider quantization or smaller imgsz
4. Proceed to P02 -- FastAPI + WebSocket backend